# M4 Final Assignment Part C02 Run MAS Locally - LangGraph

In [1]:
%pip install torch transformers accelerate bitsandbytes peft pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import pandas as pd

In [40]:
df = pd.read_csv("hitl_green_100.csv")

In [4]:
print(len(df))

100


In [5]:
print(df.columns)

Index(['doc_id', 'text', 'p_green', 'u', 'llm_green_suggested',
       'llm_confidence', 'llm_rationale', 'is_green_human', 'notes'],
      dtype='object')


In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    base_model,
    "./lora_green_patent"
)

model.eval()

print("Model ready")

c:\Users\Jonas Pedesk\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Jonas Pedesk\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.3) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:33<00:00, 11.26s/it]


Model ready


In [14]:
!pip install langchain


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import json
import re
import pandas as pd
from typing import TypedDict
from langgraph.graph import StateGraph

to run mistral locally

In [32]:
#code for local inference with Mistral + QLoRA by claude
def run_mistral(prompt: str, model, tokenizer, max_new_tokens: int = 300) -> str:
    """Generate text using the loaded Mistral + QLoRA model."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the NEW tokens (strip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Graph state

In [33]:
#code for defining the state of the patent debate by claude
class PatentState(TypedDict):
    doc_id: str
    text: str
    advocate_argument: str
    skeptic_argument: str
    is_green_tech: str       # "0" or "1"
    judge_rationale: str

agent nodes

In [34]:
#code structure assisted by claude but prompt engineering and implementation by me
def advocate_node(state: PatentState, model, tokenizer) -> PatentState:
    prompt = f"""Patent Claim: {state['text']}

Role: Advocate. Briefly explain why this patent could be environmentally beneficial.
Focus on sustainability, emissions reduction, resource efficiency, or alignment with CPC Y02 green technology.
Respond in 2-3 sentences."""

    result = run_mistral(prompt, model, tokenizer, max_new_tokens=200)
    return {**state, "advocate_argument": result}


def skeptic_node(state: PatentState, model, tokenizer) -> PatentState:
    prompt = f"""Patent Claim: {state['text']}

Role: Skeptic. Identify reasons why this claim may NOT be environmentally friendly or may involve greenwashing.
Consider potential negative environmental impacts, lack of innovation, or misalignment with CPC Y02 criteria.
Respond in 2-3 sentences."""

    result = run_mistral(prompt, model, tokenizer, max_new_tokens=200)
    return {**state, "skeptic_argument": result}


def judge_node(state: PatentState, model, tokenizer) -> PatentState:
    prompt = f"""Patent Claim: {state['text']}

Advocate argument: {state['advocate_argument']}
Skeptic argument: {state['skeptic_argument']}

You are a Judge. Consider both arguments and decide whether the patent qualifies as green technology under CPC Y02.
Return ONLY valid JSON (no markdown, no extra text):
{{"advocate_argument": "short summary", "skeptic_argument": "short summary", "is_green_tech": "0 or 1", "judge_rationale": "brief explanation"}}"""

    raw = run_mistral(prompt, model, tokenizer, max_new_tokens=300)

    # Parse JSON robustly
    try:
        # Try to extract JSON block if model adds extra text
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        parsed = json.loads(match.group()) if match else json.loads(raw)
        is_green = str(parsed.get("is_green_tech", "0")).strip()
        rationale = parsed.get("judge_rationale", raw)
    except Exception:
        # Fallback: try to infer from raw text
        is_green = "1" if any(w in raw.lower() for w in ["yes", "green", "qualifies", "environmentally"]) else "0"
        rationale = raw[:300]

    return {
        **state,
        "is_green_tech": is_green,
        "judge_rationale": rationale,
    }

build langgraph

In [43]:
from langgraph.graph import END

In [44]:
#code structure assisted by claude
def build_graph(model, tokenizer):
    """Build and compile the LangGraph MAS pipeline."""

    # Wrap nodes with model/tokenizer via closure
    def advocate(state):  return advocate_node(state, model, tokenizer)
    def skeptic(state):   return skeptic_node(state, model, tokenizer)
    def judge(state):     return judge_node(state, model, tokenizer)

    graph = StateGraph(PatentState)
    graph.add_node("advocate", advocate)
    graph.add_node("skeptic", skeptic)
    graph.add_node("judge", judge)

    graph.set_entry_point("advocate")
    graph.add_edge("advocate", "skeptic")
    graph.add_edge("skeptic", "judge")
    graph.add_edge("judge", END)

    return graph.compile()

run pipeline on all 100 claims #code assisted by claude

In [45]:
def run_mas_pipeline(df: pd.DataFrame, model, tokenizer, save_path: str = "mas_results.csv") -> pd.DataFrame:

    pipeline = build_graph(model, tokenizer)

    results = []

    for i, row in df.iterrows():
        print(f"Processing {i+1}/{len(df)} — doc_id: {row['doc_id']}")

        initial_state = PatentState(
            doc_id=str(row["doc_id"]),
            text=str(row["text"]),
            advocate_argument="",
            skeptic_argument="",
            is_green_tech="",
            judge_rationale="",
        )

        try:
            final_state = pipeline.invoke(initial_state)
        except Exception as e:
            print(f"  ⚠️  Error on doc_id {row['doc_id']}: {e}")
            final_state = {
                **initial_state,
                "advocate_argument": "ERROR",
                "skeptic_argument": "ERROR",
                "is_green_tech": "0",
                "judge_rationale": f"Error: {str(e)}",
            }

        results.append({
            "doc_id":             final_state["doc_id"],
            "text":               final_state["text"],
            "advocate_argument":  final_state["advocate_argument"],
            "skeptic_argument":   final_state["skeptic_argument"],
            "is_green_tech":      final_state["is_green_tech"],
            "judge_rationale":    final_state["judge_rationale"],
        })

    results_df = pd.DataFrame(results)
    results_df.to_csv(save_path, index=False)
    print(f"\n✅ Done! Results saved to: {save_path}")
    return results_df

In [46]:
results_df = run_mas_pipeline(
    df=df,
    model=model,
    tokenizer=tokenizer,
    save_path="mas_results.csv"
)

results_df.head()

Processing 1/100 — doc_id: 9315787
Processing 2/100 — doc_id: 9334087
Processing 3/100 — doc_id: 9686855
Processing 4/100 — doc_id: 9551706
Processing 5/100 — doc_id: 9745777
Processing 6/100 — doc_id: 9237703
Processing 7/100 — doc_id: 9273641
Processing 8/100 — doc_id: 9206804
Processing 9/100 — doc_id: 8466241
Processing 10/100 — doc_id: 8581146
Processing 11/100 — doc_id: 8975201
Processing 12/100 — doc_id: 9732596
Processing 13/100 — doc_id: 8661630
Processing 14/100 — doc_id: 9484843
Processing 15/100 — doc_id: 9703511
Processing 16/100 — doc_id: 8770118
Processing 17/100 — doc_id: 9485853
Processing 18/100 — doc_id: 8928526
Processing 19/100 — doc_id: 8607927
Processing 20/100 — doc_id: 9449733
Processing 21/100 — doc_id: 9683299
Processing 22/100 — doc_id: 9510426
Processing 23/100 — doc_id: 9787709
Processing 24/100 — doc_id: 9122206
Processing 25/100 — doc_id: 8458389
Processing 26/100 — doc_id: 9257365
Processing 27/100 — doc_id: 8423232
Processing 28/100 — doc_id: 9764760
P

,doc_id,text,advocate_argument,skeptic_argument,is_green_tech,judge_rationale
0,9315787,1. A DNA polymerase whose amino acid sequence ...,This patent for a modified DNA polymerase coul...,This claim may not be environmentally friendly...,0,While the technology may have potential benefi...
1,9334087,1. A standing pouch comprising: a first pack w...,This patent for a standing pouch could potenti...,The claim describes a standing pouch with a te...,1,While the use of exothermic or endothermic rea...
2,9686855,1. A multilayer ceramic capacitor with interpo...,This patent for a multilayer ceramic capacitor...,This claim may not be environmentally friendly...,0,While the invention may lead to more efficient...
3,9551706,1. A method of detecting differences in densit...,This patent could potentially contribute to en...,While the method described in this claim could...,0,While the technology could contribute to resou...
4,9745777,1. An electronically controlled hatch system f...,This patent for an electronically controlled h...,This claim may not be environmentally friendly...,0,While the patent may have potential for energy...
